In [1]:
import os
import datetime

folder_path = r"Z:\HTOC\HTOC Reports\I&W Reports\3. Finished I&W Reports"
two_weeks_ago = datetime.datetime.now() - datetime.timedelta(weeks=2)

recent_files = []
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):
        modified_time = datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
        if modified_time >= two_weeks_ago:
            recent_files.append(file_path)

recent_files

['Z:\\HTOC\\HTOC Reports\\I&W Reports\\3. Finished I&W Reports\\HTOC-20250609-0732-A.pdf',
 'Z:\\HTOC\\HTOC Reports\\I&W Reports\\3. Finished I&W Reports\\HTOC-20250610-0956-A - Copy.pdf',
 'Z:\\HTOC\\HTOC Reports\\I&W Reports\\3. Finished I&W Reports\\HTOC-20250610-0956-A.pdf',
 'Z:\\HTOC\\HTOC Reports\\I&W Reports\\3. Finished I&W Reports\\HTOC-20250616-1000-A.pdf']

In [36]:
import re

cleaned_groups = {}
for title, content in groups.items():
    # Remove excessive whitespace and multiple newlines
    clean_content = re.sub(r'\n\s*\n+', '\n\n', content.strip())
    cleaned_groups[title.strip()] = clean_content

cleaned_groups


{'Executive Summary': '190.92.174 [.]130 resolves to India un der ASN 199404 (WHG Hosting Services Ltd)  with reve rse DNS \nresolution  to domain of unknown reputation s3744.bom1.stableserver [.]net. 190.92.174 [.]130 was \nreported by the United States Government (USG) on 11 April 2025 as an indicator  for an unspecified \ncyber threat actor targeting federal online portals. According to the FBI and the Intelligence Community \n(IC), the cyber threat actor successfully gained unauthorized access to various federal agency online \nportals via social engineerin g. 190.92.174 [.]130 has 16 open ports, including 22/SSH, 80/HTTP , \n110/POP3, 143/IMAP, 443/HTTPS, 465/SMTP -SSL, 587/SMTP, 2077/ Old Tivoli Storage \nManage r/TrelliSoft Agent /IRLP - Internet Radio Linking Project , 2080/AutoDesk, 2086/GNUNet, and \n2095/ NBX SER /PalTalk . One security vendor labels 190.92.174 [.]130 as a malicious node in VirusTotal. \nPulsedive indicates 190.92.174 [.]130 is an unknown risk.  \n\nOn 4 and

In [10]:
sections = ['Title/Serial Number:', 'Executive Summary:', 'Technical Details:']
section_data = {}
current_section = None
lines = text.splitlines()
buffer = []

for line in lines:
    line_strip = line.strip()
    if any(line_strip.startswith(sec) for sec in sections):
        if current_section and buffer:
            section_data[current_section] = '\n'.join(buffer).strip()
            buffer = []
        for sec in sections:
            if line_strip.startswith(sec):
                current_section = sec
                break
        # Remove the section header from the line if there's content after the colon
        after_colon = line_strip[len(current_section):].strip()
        if after_colon:
            buffer.append(after_colon)
    elif current_section:
        buffer.append(line)
if current_section and buffer:
    section_data[current_section] = '\n'.join(buffer).strip()

section_data

{'Title/Serial Number:': 'HTOC—Indian IP Address  Reported by the USG as Indicator  Associated with\nUnspecified Malicious Cyber Actor Targeting USG Portals Seen in Observations for  Three  HTOC Partners, \nJune 2025 /I&W-20250 609-0732 -A \n \n \nExecutive Summary :   \n \n190.92.174 [.]130 resolves to India un der ASN 199404 (WHG Hosting Services Ltd)  with reve rse DNS \nresolution  to domain of unknown reputation s3744.bom1.stableserver [.]net. 190.92.174 [.]130 was \nreported by the United States Government (USG) on 11 April 2025 as an indicator  for an unspecified \ncyber threat actor targeting federal online portals. According to the FBI and the Intelligence Community \n(IC), the cyber threat actor successfully gained unauthorized access to various federal agency online \nportals via social engineerin g. 190.92.174 [.]130 has 16 open ports, including 22/SSH, 80/HTTP , \n110/POP3, 143/IMAP, 443/HTTPS, 465/SMTP -SSL, 587/SMTP, 2077/ Old Tivoli Storage \nManage r/TrelliSoft Agent /

In [ ]:
# Extract the indicator (IP address) from the technical details section
tech_details = cleaned_groups.get('The indicator observed at three  HTOC partner networks', '')
ip_match = re.search(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', tech_details.replace('[.]', '.'))
indicator = ip_match.group(0) if ip_match else None
print("Extracted indicator from technical details:", indicator)

None
Extracted indicator from technical details: 190.92.174.130


In [50]:
import pandas as pd

# Split tech_details into lines and find the section with indicator data
lines = tech_details.splitlines()
start_idx = None
for i, line in enumerate(lines):
    if "Indicators/Identifiers" in line:
        start_idx = i
        break

# Extract rows after the header
data_rows = []
if start_idx is not None:
    # The next 4 lines are headers, then data starts
    headers = ["Indicator", "Type", "Date Observed", "Observed By", "Notes"]
    for j in range(start_idx + 4, len(lines)):
        row = lines[j].strip()
        if not row or row.startswith("Non -Observed IOCs"):
            break
        data_rows.append(row.split("  "))  # Split by double space

    # Clean up and align columns
    df = pd.DataFrame([list(filter(None, map(str.strip, row))) for row in data_rows], columns=["Indicator", "Type"])
    # Further parsing may be needed depending on actual row structure
    # Since all information belongs to the same record, flatten and join the data_rows
    flat_row = []
    for row in data_rows:
        # Only add non-date fields (skip anything that looks like a date)
        for item in row:
            # Skip items that match date pattern (e.g., 06/04/2025)
            if not re.match(r'\d{2}/\d{2}/\d{4}', item.strip()):
                # Only add to flat_row if not a date and not in the Notes column
                # Since we want to remove 'Date Observed' and 'Notes', skip those columns
                # We'll only keep 'Indicator', 'Type', and 'Observed By'
                if not re.match(r'\d{2}/\d{2}/\d{4}', item.strip()):
                    flat_row.append(item)
    # After building flat_row, keep only the first 3 columns: Indicator, Type, Observed By
    flat_row = flat_row[:3]
    headers = ["Indicator", "Type", "Observed By"]
    # Pad with empty strings if not enough columns
    while len(flat_row) < len(headers):
        flat_row.append('')
    # Only keep as many columns as headers
    # Merge 'IPv4' and 'Address' into a single 'Type' column if present
    if 'IPv4' in flat_row and 'Address' in flat_row:
        ipv4_idx = flat_row.index('IPv4')
        if ipv4_idx + 1 < len(flat_row) and flat_row[ipv4_idx + 1] == 'Address':
            # Merge and replace
            flat_row[ipv4_idx] = 'IPv4 Address'
            del flat_row[ipv4_idx + 1]
            # Pad if needed after deletion

            # Check for observed by entities in the passage
            observed_entities = []
            for entity in ["HRSA", "NIH", "VA", "CMS", "CDC", "HHS", "OS", "FHA", "DHA", "IHS"]:
                if entity in tech_details:
                    observed_entities.append(entity)
            # Join found entities and put in the 'Observed By' column
            if observed_entities:
                flat_row[3] = " ".join(observed_entities)
            while len(flat_row) < len(headers):
                flat_row.append('')
    flat_row = flat_row[:len(headers)]
    df = pd.DataFrame([flat_row], columns=headers)
    display(df)
else:
    print("No indicator table found in tech_details.")
 pandas as pd

# Split tech_details into lines and find the section with indicator data
lines = tech_details.splitlines()
start_idx = None
for i, line in enumerate(lines):
    if "Indicators/Identifiers" in line:
        start_idx = i
        break

# Extract rows after the header
data_rows = []
if start_idx is not None:
    # The next 4 lines are headers, then data starts
    headers = ["Indicator", "Type", "Date Observed", "Observed By", "Notes"]
    for j in range(start_idx + 4, len(lines)):
        row = lines[j].strip()
        if not row or row.startswith("Non -Observed IOCs"):
            break
        data_rows.append(row.split("  "))  # Split by double space

    # Clean up and align columns
    df = pd.DataFrame([list(filter(None, map(str.strip, row))) for row in data_rows], columns=["Indicator", "Type"])
    # Further parsing may be needed depending on actual row structure
    # Since all information belongs to the same record, flatten and join the data_rows
    flat_row = []
    for row in data_rows:
        # Only add non-date fields (skip anything that looks like a date)
        for item in row:
            # Skip items that match date pattern (e.g., 06/04/2025)
            if not re.match(r'\d{2}/\d{2}/\d{4}', item.strip()):
                # Only add to flat_row if not a date and not in the Notes column
                # Since we want to remove 'Date Observed' and 'Notes', skip those columns
                # We'll only keep 'Indicator', 'Type', and 'Observed By'
                if not re.match(r'\d{2}/\d{2}/\d{4}', item.strip()):
                    flat_row.append(item)
    # After building flat_row, keep only the first 3 columns: Indicator, Type, Observed By
    flat_row = flat_row[:3]
    headers = ["Indicator", "Type", "Observed By"]
    # Pad with empty strings if not enough columns
    while len(flat_row) < len(headers):
        flat_row.append('')
    # Only keep as many columns as headers
    # Merge 'IPv4' and 'Address' into a single 'Type' column if present
    if 'IPv4' in flat_row and 'Address' in flat_row:
        ipv4_idx = flat_row.index('IPv4')
        if ipv4_idx + 1 < len(flat_row) and flat_row[ipv4_idx + 1] == 'Address':
            # Merge and replace
            flat_row[ipv4_idx] = 'IPv4 Address'
            del flat_row[ipv4_idx + 1]
            # Pad if needed after deletion

            # Check for observed by entities in the passage
            observed_entities = []
            for entity in ["HRSA", "NIH", "VA", "CMS", "CDC", "HHS", "OS", "FHA", "DHA", "IHS"]:
                if entity in tech_details:
                    observed_entities.append(entity)
            # Join found entities and put in the 'Observed By' column
            if observed_entities:
                flat_row[3] = " ".join(observed_entities)
            while len(flat_row) < len(headers):
                flat_row.append('')
    flat_row = flat_row[:len(headers)]
    df = pd.DataFrame([flat_row], columns=headers)
    display(df)
else:
    print("No indicator table found in tech_details.")


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 68)

In [24]:
import re

# Search for the pattern 'IPv4 \nAddress' in tech_details
ipv4_address_line = None
match = re.search(r'IPv4\s*\nAddress', tech_details)
if match:
    # Find the line containing 'IPv4' and the next line (which should be 'Address ...')
    lines = tech_details.splitlines()
    for i, l in enumerate(lines):
        if 'IPv4' in l:
            # Return the line with 'IPv4' and the next line
            ipv4_address_line = l + '\n' + lines[i+1] if i+1 < len(lines) else l
            break

ipv4_address_line

'190.92.174[.]130  IPv4 \nAddress  06/04/2025  '